
# 16 — Prompt Engineering for Qualitative Analysis and Classification

**Learning reference:**
[BrightPool — Prompt Engineering Course](https://github.com/BrightPool/udemy-prompt-engineering-course)

**Adapted from:**
- Qualitative analysis of interview transcripts
- Review classification with structured outputs
- Thematic labeling and classification evaluation

## Goal

Analyze customer feedback using Gemini and compare
two classification prompts.

The experiment will:

1. Create a small fictional feedback dataset.
2. Ask Gemini to discover recurring themes.
3. Classify the same messages with a baseline prompt.
4. Classify them again with an improved prompt.
5. Validate model-generated JSON.
6. Compare predictions with predefined reference labels.
7. Convert predictions into an analysis-ready table.

The improved prompt uses explicit category definitions,
a worked example and rules for ambiguous feedback.

This demonstrates how prompt design can influence
model behavior in a qualitative analysis workflow.

Three Gemini requests are planned. Evaluation and
data transformation run locally without further
model calls.



## 1. Qualitative Analysis vs. Classification

**Qualitative analysis** identifies recurring ideas
or patterns in unstructured text.

For example, several customer comments may describe
shipping problems using different words:

- "My parcel is late."
- "The package never showed up."
- "I cannot track my delivery."

A thematic analysis can group these under a common
theme such as `delivery`.

**Theme discovery** is exploratory:
the model proposes themes based on the data.

**Classification** is more constrained:
the model assigns records to predefined categories.

We will experiment with both, but only the
classification task will be scored against
reference labels.

### Prompt-engineering question

Does giving Gemini clearer category definitions,
a worked example and explicit ambiguity rules
improve its agreement with the reference labels?

A stronger prompt is not guaranteed to perform
better. We will measure the observed results.


In [1]:

%pip install -q "langchain-core>=1,<2" "langchain-openai>=1,<2" pandas


In [2]:

import json
import pandas as pd

feedback = [
    {
        "id": 1,
        "text": "My parcel still hasn't shown up.",
        "expected": "delivery"
    },
    {
        "id": 2,
        "text": "The tracking page says shipped, but "
                "nothing has arrived.",
        "expected": "delivery"
    },
    {
        "id": 3,
        "text": "The headphones stopped working "
                "after two days.",
        "expected": "product_quality"
    },
    {
        "id": 4,
        "text": "The bottle cap broke the first "
                "time I used it.",
        "expected": "product_quality"
    },
    {
        "id": 5,
        "text": "The mobile app freezes when "
                "I try to check out.",
        "expected": "app_experience"
    },
    {
        "id": 6,
        "text": "The website keeps logging me out "
                "while I'm shopping.",
        "expected": "app_experience"
    },
    {
        "id": 7,
        "text": "The subscription costs more "
                "than I expected.",
        "expected": "pricing"
    },
    {
        "id": 8,
        "text": "Why did the monthly plan become "
                "so expensive?",
        "expected": "pricing"
    },
    {
        "id": 9,
        "text": "The delivery was late, and "
                "the box was crushed, but the "
                "product works perfectly.",
        "expected": "delivery"
    },
    {
        "id": 10,
        "text": "The app looks nice, but the "
                "subscription is too expensive.",
        "expected": "pricing"
    }
]

df = pd.DataFrame(feedback)

print(df.to_string(index=False))
print("\nFeedback records:", len(df))

assert len(df) == 10
assert df["id"].is_unique


 id                                                                             text        expected
  1                                                 My parcel still hasn't shown up.        delivery
  2                         The tracking page says shipped, but nothing has arrived.        delivery
  3                                   The headphones stopped working after two days. product_quality
  4                                   The bottle cap broke the first time I used it. product_quality
  5                                  The mobile app freezes when I try to check out.  app_experience
  6                             The website keeps logging me out while I'm shopping.  app_experience
  7                                     The subscription costs more than I expected.         pricing
  8                                    Why did the monthly plan become so expensive?         pricing
  9 The delivery was late, and the box was crushed, but the product works perfectly.       


## 2. Reference Labels and Evaluation Design

We define five possible categories:

- `delivery`: Shipping, tracking or arrival issues.
- `product_quality`: Broken, defective or unreliable products.
- `app_experience`: Application or website functionality.
- `pricing`: Prices, charges or subscription cost.
- `other`: Feedback outside these categories.

Each record receives exactly one primary category.

The `expected` column contains labels assigned
before the model is called. These serve as the
reference answers for our experiment.

Two records contain multiple observations:

**Record 9:** A late delivery and a crushed box,
but a functioning product. We prioritize the
delivery problem.

**Record 10:** A positive app observation and a
negative pricing complaint. We prioritize pricing.

The improved prompt will explain these decision
rules using general instructions rather than
revealing the reference answers for test records.

### Evaluation metrics

- Classification accuracy
- Number of invalid category labels
- Number of missing or duplicate record IDs
- Prediction changes between prompts

Because the dataset is small and synthetic, the
results are illustrative rather than a reliable
estimate of real-world accuracy.


In [3]:

from getpass import getpass

from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    SystemMessage,
    HumanMessage
)

MODEL = "gemini-3.5-flash-lite"

gemini_key = getpass("Enter your Gemini API key: ")

gemini = ChatOpenAI(
    model=MODEL,
    api_key=gemini_key,
    base_url=(
        "https://generativelanguage.googleapis.com/"
        "v1beta/openai/"
    ),
    temperature=0,
    max_retries=0,
    timeout=120,
    use_responses_api=False
)


def call_json_model(system_prompt, user_prompt):
    response = gemini.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ])

    text = response.content.strip()

    # Some models wrap JSON in Markdown fences.
    if text.startswith("```"):
        lines = text.splitlines()

        if lines[-1].strip() == "```":
            text = "\n".join(lines[1:-1]).strip()

    return json.loads(text)


def feedback_as_json():
    records = [
        {"id": row["id"], "text": row["text"]}
        for row in feedback
    ]

    return json.dumps(records, indent=2)


print("Gemini configured:", MODEL)
print("JSON response parser ready.")
print("Model requests: 0")


Enter your Gemini API key: ··········
Gemini configured: gemini-3.5-flash-lite
JSON response parser ready.
Model requests: 0



## 3. Discover Themes with Gemini

The first task is exploratory.

Gemini receives all ten feedback messages and
identifies recurring customer concerns.

The prompt asks for:

- Short, descriptive theme names.
- A brief explanation of each theme.
- IDs of messages supporting each theme.

A single feedback message may support more than
one theme in this exploratory stage.

This differs from the later classification task,
which requires exactly one primary category.

The discovered themes are observations generated
by the model, not verified ground truth.

We will inspect them before comparing the two
classification prompts.


In [4]:

theme_system_prompt = """
You are a qualitative research analyst.

Identify recurring customer concerns in the
provided feedback messages.

Return valid JSON only, using this structure:

{
  "themes": [
    {
      "name": "short_theme_name",
      "description": "What this theme represents",
      "sample_ids": [1, 2]
    }
  ]
}

Rules:
- Identify 3 to 5 meaningful themes.
- Use short lowercase snake_case names.
- Use only information in the feedback.
- Include only real sample IDs.
- Each theme must have supporting samples.
- One sample may belong to multiple themes.
- Do not invent customer experiences.
- Do not include Markdown formatting.
"""

theme_user_prompt = (
    "Analyze these customer feedback records:\n\n"
    + feedback_as_json()
)

if "theme_result" not in globals():
    theme_result = call_json_model(
        theme_system_prompt,
        theme_user_prompt
    )
    print("Gemini theme discovery completed.")
else:
    print("Using saved theme result.")

assert isinstance(theme_result, dict)
assert isinstance(theme_result.get("themes"), list)

valid_ids = set(df["id"])

for theme in theme_result["themes"]:
    assert isinstance(theme["name"], str)
    assert isinstance(theme["sample_ids"], list)
    assert theme["sample_ids"]
    assert set(theme["sample_ids"]) <= valid_ids

print("\nDiscovered themes:")

for theme in theme_result["themes"]:
    print("\nTheme:", theme["name"])
    print("Description:", theme["description"])
    print("Supporting IDs:", theme["sample_ids"])


Gemini theme discovery completed.

Discovered themes:

Theme: delivery_delays
Description: Issues related to parcels not showing up or arriving late.
Supporting IDs: [1, 2, 9]

Theme: product_quality
Description: Products breaking quickly or stopping working shortly after purchase.
Supporting IDs: [3, 4]

Theme: app_website_issues
Description: Technical problems including mobile app freezes and unexpected website logouts.
Supporting IDs: [5, 6]

Theme: pricing_concerns
Description: Complaints about subscriptions costing more than expected or being too expensive.
Supporting IDs: [7, 8, 10]



## 4. Baseline Classification Prompt

Next, we move from open-ended theme discovery
to a fixed classification task.

The baseline prompt provides:

- The allowed category names.
- A basic classification instruction.
- The required JSON output format.

It does not explain how to handle ambiguous
feedback or provide examples of the categories.

All ten messages are sent in one model request.

The baseline produces predictions that we can
compare with the more detailed prompt later.

Importantly, both prompts receive the same input
records and use the same Gemini model.


In [5]:

baseline_system_prompt = """
Classify each customer feedback message into
one of these categories:

delivery
product_quality
app_experience
pricing
other

Choose one category per message.

Return valid JSON only:

{
  "labels": [
    {"id": 1, "category": "delivery"}
  ]
}

Include all input IDs.
Do not include explanations or Markdown.
"""

classification_user_prompt = (
    "Classify the following feedback:\n\n"
    + feedback_as_json()
)

if "baseline_result" not in globals():
    baseline_result = call_json_model(
        baseline_system_prompt,
        classification_user_prompt
    )
    print("Baseline classification completed.")
else:
    print("Using saved baseline predictions.")

print(json.dumps(baseline_result, indent=2))


Baseline classification completed.
{
  "labels": [
    {
      "id": 1,
      "category": "delivery"
    },
    {
      "id": 2,
      "category": "delivery"
    },
    {
      "id": 3,
      "category": "product_quality"
    },
    {
      "id": 4,
      "category": "product_quality"
    },
    {
      "id": 5,
      "category": "app_experience"
    },
    {
      "id": 6,
      "category": "app_experience"
    },
    {
      "id": 7,
      "category": "pricing"
    },
    {
      "id": 8,
      "category": "pricing"
    },
    {
      "id": 9,
      "category": "delivery"
    },
    {
      "id": 10,
      "category": "pricing"
    }
  ]
}



## 5. Improve the Classification Prompt

The baseline prompt is short, but several
classification decisions can be ambiguous.

The improved prompt adds:

**Category definitions**

Each label has an explicit meaning.

**Primary issue rule**

When a message contains multiple observations,
classify the main problem rather than every issue.

**Negative issue prioritization**

When a message includes praise and a complaint,
classify the complaint.

**Clarification of product damage**

Packaging damage does not automatically mean
a product is defective.

**Few-shot example**

One worked example demonstrates the expected
category and JSON structure.

**Output constraints**

Each ID must appear exactly once, and each category
must match an allowed label.

We are testing whether these additional instructions
improve agreement with the predefined reference labels.

The example in the prompt is separate from the
ten evaluation records.


In [6]:

improved_system_prompt = """
You are a customer-feedback classification specialist.

Assign exactly ONE PRIMARY category to each message.

ALLOWED CATEGORIES

delivery:
Shipping, parcels, tracking, late arrival,
missing packages or delivery-related problems.

product_quality:
Broken, defective, unreliable or malfunctioning
products. Do not classify packaging damage alone
as product_quality when the product works correctly.

app_experience:
Technical problems with the website or mobile app,
including crashes, login issues, and checkout errors.

pricing:
Complaints or questions about prices, charges,
fees, subscription costs or affordability.

other:
Messages whose primary issue is not covered above.

DECISION RULES

1. Identify the customer's main concern.
2. If the message contains praise and a complaint,
   classify the complaint.
3. If multiple problems are mentioned, select the
   main actionable issue expressed by the customer.
4. Packaging damage without product malfunction
   is not automatically a product_quality issue.
5. Never invent a category.
6. Never assign more than one label per record.

EXAMPLE

Input:
{"id": 99,
 "text": "I like the design, but the membership fee is too high."}

Output:
{"id": 99, "category": "pricing"}

RESPONSE FORMAT

Return one valid JSON object:

{
  "labels": [
    {"id": 1, "category": "delivery"}
  ]
}

Return every input ID exactly once.
Return only JSON, without explanations or Markdown.
"""

if "improved_result" not in globals():
    improved_result = call_json_model(
        improved_system_prompt,
        classification_user_prompt
    )
    print("Improved classification completed.")
else:
    print("Using saved improved predictions.")

print(json.dumps(improved_result, indent=2))


Improved classification completed.
{
  "labels": [
    {
      "id": 1,
      "category": "delivery"
    },
    {
      "id": 2,
      "category": "delivery"
    },
    {
      "id": 3,
      "category": "product_quality"
    },
    {
      "id": 4,
      "category": "product_quality"
    },
    {
      "id": 5,
      "category": "app_experience"
    },
    {
      "id": 6,
      "category": "app_experience"
    },
    {
      "id": 7,
      "category": "pricing"
    },
    {
      "id": 8,
      "category": "pricing"
    },
    {
      "id": 9,
      "category": "delivery"
    },
    {
      "id": 10,
      "category": "pricing"
    }
  ]
}



## 6. Evaluate the Two Prompts

We will compare both sets of predictions against
the predefined reference labels.

Validation checks:

- The response has the expected `labels` list.
- Every entry contains a valid ID and category.
- No record ID appears more than once.
- No records are missing.
- No unrecognized categories are used.

If a response is malformed, the corresponding
missing or invalid predictions count as incorrect.

We will compare:

1. Baseline accuracy.
2. Improved-prompt accuracy.
3. Which records changed classification.
4. Whether those changes were correct.

This evaluation is deterministic Python code.

No additional Gemini requests are needed.


In [7]:

ALLOWED = {
    "delivery",
    "product_quality",
    "app_experience",
    "pricing",
    "other"
}

expected = dict(zip(df["id"], df["expected"]))


def extract_predictions(result):
    predictions = {}
    problems = []

    labels = result.get("labels", [])

    if not isinstance(labels, list):
        return {}, ["labels is not a list"]

    for item in labels:
        if not isinstance(item, dict):
            problems.append("Non-object label entry")
            continue

        record_id = item.get("id")
        category = item.get("category")

        if type(record_id) is not int:
            problems.append(f"Invalid ID: {record_id!r}")
            continue

        if record_id not in expected:
            problems.append(f"Unknown ID: {record_id}")
            continue

        if record_id in predictions:
            problems.append(f"Duplicate ID: {record_id}")
            predictions[record_id] = None
            continue

        if category not in ALLOWED:
            problems.append(
                f"Invalid category for ID {record_id}: "
                f"{category!r}"
            )
            predictions[record_id] = None
            continue

        predictions[record_id] = category

    missing = set(expected) - set(predictions)

    if missing:
        problems.append(f"Missing IDs: {sorted(missing)}")

    return predictions, problems


baseline_predictions, baseline_problems = (
    extract_predictions(baseline_result)
)

improved_predictions, improved_problems = (
    extract_predictions(improved_result)
)


comparison = df.copy()

comparison["baseline"] = comparison["id"].map(
    baseline_predictions
)

comparison["improved"] = comparison["id"].map(
    improved_predictions
)

comparison["baseline_correct"] = (
    comparison["baseline"] == comparison["expected"]
)

comparison["improved_correct"] = (
    comparison["improved"] == comparison["expected"]
)

baseline_score = int(
    comparison["baseline_correct"].sum()
)

improved_score = int(
    comparison["improved_correct"].sum()
)

total = len(comparison)

print(
    comparison[
        [
            "id",
            "expected",
            "baseline",
            "improved",
            "baseline_correct",
            "improved_correct"
        ]
    ].to_string(index=False)
)

print("\nBASELINE")
print(f"Accuracy: {baseline_score}/{total}")
print("Validation issues:", baseline_problems or "None")

print("\nIMPROVED PROMPT")
print(f"Accuracy: {improved_score}/{total}")
print("Validation issues:", improved_problems or "None")

print("\nAccuracy difference:")
print(
    f"{(improved_score - baseline_score) / total:+.0%}"
)

changed = comparison[
    comparison["baseline"] != comparison["improved"]
]

print("\nPredictions changed:", len(changed))

if not changed.empty:
    print(
        changed[
            ["id", "expected", "baseline", "improved"]
        ].to_string(index=False)
    )

print("\nAdditional Gemini requests: 0")


 id        expected        baseline        improved  baseline_correct  improved_correct
  1        delivery        delivery        delivery              True              True
  2        delivery        delivery        delivery              True              True
  3 product_quality product_quality product_quality              True              True
  4 product_quality product_quality product_quality              True              True
  5  app_experience  app_experience  app_experience              True              True
  6  app_experience  app_experience  app_experience              True              True
  7         pricing         pricing         pricing              True              True
  8         pricing         pricing         pricing              True              True
  9        delivery        delivery        delivery              True              True
 10         pricing         pricing         pricing              True              True

BASELINE
Accuracy: 10/10
Valida

In [8]:

# Use the improved prompt's primary category labels.
# Invalid or missing labels remain unassigned.

one_hot = pd.DataFrame({
    category: (
        comparison["improved"] == category
    ).astype(int)
    for category in sorted(ALLOWED)
})

analysis_table = pd.concat(
    [
        comparison[
            ["id", "text", "expected", "improved"]
        ].reset_index(drop=True),
        one_hot.reset_index(drop=True)
    ],
    axis=1
)

print("Analysis-ready table:")
print(analysis_table.to_string(index=False))

print("\nCategory totals:")
print(
    analysis_table[sorted(ALLOWED)]
    .sum()
    .sort_values(ascending=False)
)

print("\nEach row has one category:")
row_totals = one_hot.sum(axis=1)
print(row_totals.to_list())

if (row_totals == 1).all():
    print("All records received one valid category.")
else:
    print(
        "Some records have no valid category. "
        "Inspect the model output."
    )

print("\nAdditional Gemini requests: 0")


Analysis-ready table:
 id                                                                             text        expected        improved  app_experience  delivery  other  pricing  product_quality
  1                                                 My parcel still hasn't shown up.        delivery        delivery               0         1      0        0                0
  2                         The tracking page says shipped, but nothing has arrived.        delivery        delivery               0         1      0        0                0
  3                                   The headphones stopped working after two days. product_quality product_quality               0         0      0        0                1
  4                                   The bottle cap broke the first time I used it. product_quality product_quality               0         0      0        0                1
  5                                  The mobile app freezes when I try to check out.  app_experien


## Findings and Limitations

### What This Notebook Demonstrates

- Qualitative theme discovery from unstructured text.
- Prompt-based customer feedback classification.
- Baseline and improved prompt comparison.
- Category definitions and ambiguity rules.
- Few-shot prompting.
- JSON response parsing and local validation.
- Evaluation against predefined reference labels.
- Conversion of classification results to
  an analysis-ready, one-hot-encoded table.

### Prompt Engineering Techniques

**Task definition**

The prompts specify whether Gemini should discover
themes or classify records into known categories.

**Explicit taxonomy**

The improved classification prompt explains each
permitted category.

**Few-shot prompting**

A worked example demonstrates the expected
classification behavior and output structure.

**Ambiguity handling**

The improved prompt provides rules for messages
that contain multiple issues or both praise
and complaints.

**Structured output instructions**

The model is asked to return a specific JSON
structure. Python validates the output.

**Prompt evaluation**

The baseline and improved prompts are tested
against the same ten messages and reference labels.

### Relationship to the Udemy Examples

This notebook combines two course ideas:

1. Thematic labeling of unstructured feedback.
2. Structured review classification and evaluation.

Unlike the original large-batch thematic analysis
workflow, this example uses a small local dataset.

It does not reproduce the course's separate
decision-model comparison, asynchronous processing
or external spreadsheet integration.


### Observed Results

Three Gemini requests were completed successfully:
one for exploratory theme discovery and two for
classification using different prompts.

**Theme discovery**

Gemini identified four recurring themes:

- `delivery_delays` — IDs 1, 2, 9
- `product_quality` — IDs 3, 4
- `app_website_issues` — IDs 5, 6
- `pricing_concerns` — IDs 7, 8, 10

**Classification comparison**

| Metric | Baseline | Improved |
|---|---:|---:|
| Correct predictions | 10/10 | 10/10 |
| Accuracy | 100% | 100% |
| Invalid or missing labels | 0 | 0 |

No predictions changed between the two prompts.

The improved prompt introduced explicit category
definitions, ambiguity rules and a worked example,
but produced no measurable accuracy improvement
on this dataset.

**Data transformation**

All ten feedback records received one valid primary
category and were converted into one-hot-encoded
columns.

Category distribution:

- Delivery: 3
- Pricing: 3
- App experience: 2
- Product quality: 2
- Other: 0

**Interpretation**

The baseline prompt was sufficient for these
ten synthetic examples.

The experiment does not establish whether the
improved prompt would perform better on a larger
or more ambiguous dataset.

More detailed instructions are not automatically
more effective; their value should be measured
against the task and evaluation criteria.




### Limitations

- The dataset contains only ten fictional records.
- The reference labels were manually predefined.
- Some feedback could reasonably fit more than
  one category.
- The improved prompt's rules reflect the selected
  annotation policy, not a universal definition.
- The model is tested only once per prompt.
- Small differences may be due to model variability.
- The prompt instructs JSON output, but does not
  use a provider-enforced JSON schema.
- Accuracy checks do not measure the usefulness
  of the discovered themes.
- This experiment does not test larger-scale
  batch processing or production monitoring.
- The results cannot establish general
  classification reliability.

### Conclusion

This notebook demonstrates how prompt engineering
can support qualitative analysis and structured
classification.

Gemini first identifies recurring themes from
customer feedback.

The same model then classifies the records using
two different prompts.

Local evaluation compares the predictions against
predefined labels without additional model calls.

The central lesson is that classification quality
depends not only on the model but also on how the
task, categories, ambiguity rules and output
requirements are communicated.

Actual results must be examined before concluding
whether a more detailed prompt is beneficial.
